# Menebak Kualitas Wine Merah dengan Random Forest

Notebook ini mencoba menjawab satu pertanyaan sederhana. Kalau kita hanya tahu kandungan kimia sebuah wine merah, bisakah kita menebak skor kualitas yang diberikan para penilai?

Datanya berisi 1.599 wine merah. Tiap wine punya 11 hasil ukur kimia dan satu skor kualitas dari penilai. Kita akan memuat data, mengenalinya, membersihkannya, melihat grafiknya, melatih sebuah model (alat penebak skor), menilai hasilnya, lalu menarik kesimpulan. Jalankan sel dari atas ke bawah, satu per satu.

## 1. Memuat data

Berkas `winequality-red.csv` kita unggah ke Colab. Setelah sel kedua di bawah dijalankan, akan muncul tombol untuk memilih berkas. Pilih berkas CSV dari komputer lalu tunggu sampai selesai.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as pe
from matplotlib.ticker import FuncFormatter, MaxNLocator
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Gaya grafik dan warna yang dipakai di seluruh notebook
sns.set_theme(style="whitegrid", rc={
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e8e8e5", "axes.edgecolor": "#c9c9c5",
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 13,
})
BIRU, ORANYE, ABU = "#2a78d6", "#eb6834", "#b8b8b4"

# Nama kolom dalam bahasa sehari-hari, dipakai di judul dan label grafik
NAMA = {
    "fixed acidity": "asam tetap", "volatile acidity": "asam mudah menguap",
    "citric acid": "asam sitrat", "residual sugar": "sisa gula",
    "chlorides": "kadar garam", "free sulfur dioxide": "belerang bebas",
    "total sulfur dioxide": "total belerang", "density": "kepekatan",
    "pH": "tingkat asam (pH)", "sulphates": "sulfat", "alcohol": "kadar alkohol",
}

def koma(angka, desimal=2):
    """Tulis angka dengan koma desimal."""
    return f"{angka:.{desimal}f}".replace(".", ",")

In [ ]:
if not os.path.exists("winequality-red.csv"):
    from google.colab import files
    files.upload()   # pilih berkas winequality-red.csv

data_mentah = pd.read_csv("winequality-red.csv", sep=None, engine="python")
data_mentah.head()

## 2. Mengenali data

Sebelum melakukan apa pun, kita lihat dulu isinya. Ada berapa baris dan kolom, apa saja namanya, dan apakah ada data yang kosong atau tercatat dua kali.

In [ ]:
print("Jumlah baris :", data_mentah.shape[0])
print("Jumlah kolom :", data_mentah.shape[1])
print("Data kosong  :", data_mentah.isna().sum().sum())
print("Baris kembar :", data_mentah.duplicated().sum())
print("Nama kolom   :", ", ".join(data_mentah.columns))
print()
data_mentah.info()

Nama kolomnya berbahasa Inggris, jadi ini arti masing-masing dalam bahasa sehari-hari.

| Kolom | Artinya |
|---|---|
| `fixed acidity` | Asam yang tidak mudah menguap. Memberi rasa asam yang tajam dan segar. |
| `volatile acidity` | Asam yang mudah menguap, terutama asam cuka. |
| `citric acid` | Asam sitrat, jenis asam yang sama dengan di jeruk lemon. Memberi rasa segar. |
| `residual sugar` | Gula yang masih tersisa setelah wine selesai dibuat. |
| `chlorides` | Jumlah garam dalam wine. |
| `free sulfur dioxide` | Belerang pengawet yang masih aktif menjaga wine. |
| `total sulfur dioxide` | Seluruh belerang pengawet dalam wine, yang aktif maupun yang sudah terikat. |
| `density` | Seberapa pekat atau berat wine dibandingkan air. |
| `pH` | Tingkat asam secara keseluruhan. Angka yang lebih kecil berarti lebih asam. |
| `sulphates` | Zat tambahan (sulfat) yang membantu wine awet. |
| `alcohol` | Kadar alkohol dalam persen. |
| `quality` | Skor kualitas dari para penilai, bernilai 3 sampai 8 di data ini. Skor inilah yang ingin kita tebak. |

Perhatikan angka "Baris kembar" di atas. Kalau tidak nol, ada wine yang datanya sama persis dan tercatat lebih dari sekali. Kita urus itu di langkah berikutnya.

## 3. Membersihkan data

Ada dua hal yang kita bereskan.

Pertama, baris kembar. Wine yang semua angkanya sama persis kemungkinan tercatat dua kali. Kalau dibiarkan, satu salinan bisa dipakai model untuk belajar sementara salinan kembarnya dipakai untuk mengujinya. Model jadi seperti murid yang diuji dengan soal yang sudah ia hafal jawabannya, sehingga nilainya tampak lebih bagus dari kemampuan aslinya. Kita simpan satu salinan saja.

In [ ]:
data = data_mentah.drop_duplicates().reset_index(drop=True)
print(f"Jumlah baris sebelum: {len(data_mentah)}, sesudah: {len(data)}")

Kedua, kolom yang kurang berperan. Tidak semua dari 11 kolom ikut menentukan skor. Grafik di bawah mengukur seberapa erat tiap kolom berhubungan dengan skor kualitas, dari 0 (sama sekali tidak berhubungan) sampai 1 (selalu naik turun bersama skor). Kolom dengan batang di bawah garis batas 0,1 kita anggap hampir tidak ikut menentukan skor, jadi kita buang. Angka 0,1 adalah patokan yang lazim untuk hubungan yang sangat lemah.

In [ ]:
# Seberapa erat tiap kolom berhubungan dengan skor kualitas
keeratan = data.corr()["quality"].drop("quality").abs().sort_values()
BATAS = 0.1
dibuang = keeratan[keeratan < BATAS].index.tolist()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([NAMA[k] + (" (dibuang)" if k in dibuang else "") for k in keeratan.index],
        keeratan.values, height=0.65,
        color=[ABU if k in dibuang else BIRU for k in keeratan.index])
ax.axvline(BATAS, color=ORANYE, linestyle="--", linewidth=1.5, label="garis batas 0,1")
for i, nilai in enumerate(keeratan.values):
    ax.text(nilai + 0.008, i, koma(nilai), va="center", fontsize=10)
ax.set(title="Seberapa erat tiap kolom berhubungan dengan skor kualitas",
       xlabel="Keeratan hubungan (makin panjang makin erat)")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

data_bersih = data.drop(columns=dibuang)
print("Kolom yang dibuang:", ", ".join(dibuang))

Tiga kolom yang terbuang adalah sisa gula, belerang bebas, dan tingkat asam (pH). Wine yang skornya tinggi maupun rendah punya sisa gula yang berkisar di angka yang sama, sekitar 2,5 sampai 2,7, jadi kolom ini tidak membantu membedakan keduanya. Belerang bebas naik turun tanpa arah yang jelas, sedangkan pH hanya bergeser tipis dari sekitar 3,4 ke 3,3 antara skor terendah dan tertinggi. Pada bagian evaluasi nanti kita buktikan bahwa membuang ketiganya tidak membuat tebakan lebih buruk.

## 4. Info data yang sudah bersih

Setelah dibersihkan, kita lihat lagi ukuran dan isi datanya. Kita juga lihat berapa banyak wine di tiap skor.

In [ ]:
print("Jumlah baris :", data_bersih.shape[0])
print("Jumlah kolom :", data_bersih.shape[1])
print("Nama kolom   :", ", ".join(data_bersih.columns))
print()
data_bersih.info()

In [ ]:
jumlah = data_bersih["quality"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(jumlah.index, jumlah.values, color=BIRU, width=0.65)
for skor, n in jumlah.items():
    ax.text(skor, n + 12, f"{n} wine\n({n / jumlah.sum():.0%})", ha="center", fontsize=10)
ax.set(title="Jumlah wine untuk tiap skor kualitas", xlabel="Skor kualitas", ylabel="Jumlah wine")
ax.margins(y=0.2)
plt.tight_layout()
plt.show()

Hampir semua wine berskor 5 atau 6. Skor 3 dan skor 8 hanya segelintir. Ketimpangan ini akan memengaruhi cara model belajar, dan kita bahas di bagian pemodelan.

## 5. Mengenal delapan kolom yang dipakai

Tiap kolom punya satu grafik, dan semuanya menjawab pertanyaan yang sama. Wine yang mendapat skor tinggi biasanya punya kandungan ini lebih banyak atau lebih sedikit?

Garis biru adalah rata-rata kandungan wine pada tiap skor, sedangkan garis putus-putus abu-abu adalah rata-rata semua wine. Angka kecil di bawah skor menunjukkan berapa wine yang ada di kelompok itu. Skor 3 dan skor 8 contohnya sedikit, jadi bentuk garis di dua ujung itu jangan terlalu dipegang.

In [ ]:
def gambar_rata_rata(kolom, satuan, desimal=2):
    """Garis rata-rata sebuah kolom di tiap skor kualitas."""
    rata = data_bersih.groupby("quality")[kolom].mean()
    umum = data_bersih[kolom].mean()
    banyak = data_bersih["quality"].value_counts()

    fig, ax = plt.subplots(figsize=(8.2, 4))
    ax.plot(rata.index, rata.values, marker="o", markersize=9, linewidth=2.5, color=BIRU)
    ax.axhline(umum, color=ABU, linestyle="--", linewidth=1.5)
    ax.annotate("rata-rata\nsemua wine", xy=(1, umum), xycoords=("axes fraction", "data"),
                xytext=(6, 0), textcoords="offset points", ha="left", va="center",
                color="#52514e", fontsize=10, annotation_clip=False)
    for skor, nilai in rata.items():
        ax.annotate(koma(nilai, desimal), (skor, nilai), xytext=(0, 11),
                    textcoords="offset points", ha="center", fontsize=11, fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=3, foreground="white")])
    ax.set_xticks(rata.index)
    ax.set_xticklabels([f"{s}\n({banyak[s]} wine)" for s in rata.index])
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: koma(v, desimal)))
    ax.set(title=f"Rata-rata {NAMA[kolom]} di tiap skor kualitas",
           xlabel="Skor kualitas (makin besar makin bagus)", ylabel=satuan)
    ax.margins(y=0.25)
    plt.tight_layout(rect=(0, 0, 0.87, 1))
    plt.show()

### Kadar alkohol

In [ ]:
gambar_rata_rata("alcohol", "persen volume (%)", 1)

Inilah kolom yang hubungannya paling kuat dengan skor. Dari skor 5 sampai skor 8, kadar alkohol naik terus, dari sekitar 9,9% menjadi sekitar 12,2%. Wine yang dinilai bagus di data ini cenderung lebih kuat daripada wine yang dinilai biasa saja. Alkohol yang cukup biasanya membuat wine terasa lebih penuh dan hangat di mulut, dan mungkin itu yang disukai para penilai.

### Asam mudah menguap

In [ ]:
gambar_rata_rata("volatile acidity", "gram per liter", 2)

Kolom ini bekerja berlawanan dengan alkohol. Makin tinggi skornya, makin sedikit asam mudah menguap, dari sekitar 0,88 pada skor 3 sampai sekitar 0,40 pada skor 7. Asam jenis ini pada dasarnya asam cuka. Sedikit saja tidak masalah, tetapi kalau kebanyakan, bau wine jadi menusuk dan rasanya mirip cuka, sehingga wajar kalau penilai memberi skor rendah. Dari delapan kolom, inilah yang paling kuat menandai wine yang kurang enak.

### Sulfat

In [ ]:
gambar_rata_rata("sulphates", "gram per liter", 2)

Sulfat adalah zat tambahan yang melindungi wine dari bakteri dan dari kerusakan karena terkena udara, sehingga wine lebih awet. Garisnya naik dari kiri ke kanan tanpa pernah turun, dari sekitar 0,57 di skor 3 sampai sekitar 0,77 di skor 8. Wine yang terlindungi dengan baik cenderung lebih segar dan stabil rasanya, dan pola itu terbaca jelas di grafik.

### Asam sitrat

In [ ]:
gambar_rata_rata("citric acid", "gram per liter", 2)

Asam sitrat adalah asam yang sama dengan yang ada di jeruk lemon. Dalam jumlah kecil, ia memberi rasa segar pada wine. Wine skor 3 dan 4 hanya mengandung sekitar 0,17, sedangkan wine skor 7 dan 8 mengandung sekitar 0,37 sampai 0,38, lebih dari dua kali lipatnya. Para penilai tampaknya menyukai wine yang punya kesegaran asam.

### Kepekatan

In [ ]:
gambar_rata_rata("density", "gram per cm³", 4)

Kepekatan mengukur seberapa berat wine dibandingkan air. Alkohol lebih ringan daripada air, jadi wine yang beralkohol tinggi cenderung lebih ringan, dan sebagian pola di grafik ini sebenarnya ikut kadar alkohol. Garisnya turun dari sekitar 0,9975 pada skor 3 menjadi sekitar 0,9951 pada skor 8, dengan sedikit goyangan di tengah. Selisihnya kecil, tetapi arahnya searah dengan wine yang lebih bagus, jadi kolom ini tetap kita pakai sebagai pelengkap.

### Total belerang

In [ ]:
gambar_rata_rata("total sulfur dioxide", "miligram per liter", 1)

Belerang dipakai sebagai pengawet. Kalau terlalu banyak, baunya menyengat dan mengganggu aroma wine. Polanya sedikit berbeda dari kolom sebelumnya. Wine skor 5 punya belerang paling banyak, sekitar 57, lalu turun menjadi sekitar 41 pada skor 6 dan sekitar 35 pada skor 7 dan 8. Jadi kolom ini terutama membantu membedakan wine skor 5 dari wine yang lebih bagus. Belerang di skor 3 dan 4 memang lebih sedikit, tetapi contohnya hanya 10 dan 53 wine, jadi tidak bisa terlalu diandalkan.

### Kadar garam

In [ ]:
gambar_rata_rata("chlorides", "gram per liter", 3)

Kolom ini mengukur jumlah garam dalam wine. Wine yang terlalu asin rasanya kurang enak, dan grafiknya sejalan dengan itu. Wine skor 3 punya sekitar 0,122, sementara wine skor 8 hanya sekitar 0,068, kira-kira separuhnya. Garisnya turun pelan dengan satu goyangan kecil di skor 5. Pengaruhnya nyata, tetapi tidak sebesar alkohol dan asam cuka.

### Asam tetap

In [ ]:
gambar_rata_rata("fixed acidity", "gram per liter", 1)

Asam tetap adalah asam yang tidak menguap, yang memberi rasa asam tajam dan segar pada wine. Dari delapan kolom, inilah yang polanya paling samar. Garisnya naik turun. Wine skor 7 rata-rata sedikit lebih tinggi (sekitar 8,9) daripada wine skor 4 (sekitar 7,8), tetapi tidak membentuk tangga yang rapi. Kolom ini kita pertahankan karena hubungannya dengan skor masih sedikit di atas batas 0,1, dan perannya kecil sebagai pelengkap kolom lain.

## 6. Pemodelan

### Kenapa Random Forest?

Bayangkan kita mengundang 300 juri pencicip wine. Tiap juri hanya diberi sebagian contoh wine dan sebagian ciri untuk dipelajari, sehingga pengalaman mereka berbeda-beda. Ketika ada wine baru, setiap juri menebak skornya sendiri, lalu skor yang paling banyak dipilih menjadi jawaban akhir. Itulah Random Forest, atau "hutan acak". Satu juri bisa salah, tetapi suara 300 juri dengan pengalaman berbeda biasanya jauh lebih bisa dipercaya.

Cara ini cocok untuk data wine kita karena beberapa hal.

- Datanya berupa tabel angka hasil ukur kimia, dan Random Forest memang kuat untuk data seperti ini.
- Ukuran angka tidak jadi soal. Total belerang bernilai puluhan sedangkan kepekatan bernilai sekitar satu, dan keduanya bisa langsung dimasukkan tanpa diubah dulu.
- Pola yang tidak lurus bisa ditangkap, seperti total belerang yang puncaknya di skor 5 lalu turun.
- Kita bisa tahu kolom mana yang paling berpengaruh, jadi hasilnya bisa dijelaskan dan tidak sekadar tebakan.
- Pada laporan proyek kuliah ESE417 (Wu dan Zhang, 2023) yang memakai data wine merah yang sama, Random Forest memberi hasil terbaik dibanding tiga cara menebak lain yang mereka coba.

### Dua versi model

Seperti terlihat di langkah 4, wine skor 5 dan 6 jauh lebih banyak daripada skor lain. Juri-juri tadi bisa jadi malas. Mereka belajar bahwa menebak "5 atau 6" hampir selalu benar, lalu mengabaikan skor yang jarang muncul. Karena itu kita siapkan dua versi.

- Model biasa belajar apa adanya.
- Model seimbang diberi pesan tambahan agar lebih memperhatikan wine dari skor yang jarang muncul. Salah menebak wine skor langka dihitung lebih berat daripada salah menebak wine skor 5 atau 6.

### Cara mengujinya

Menguji model dengan wine yang sudah ia pelajari tidak adil. Karena itu data dibagi menjadi 5 bagian. Tiap giliran, satu bagian disembunyikan, model belajar dari empat bagian lainnya, lalu menebak bagian yang disembunyikan tadi. Setelah lima giliran, setiap wine sudah pernah ditebak tanpa model pernah melihat jawabannya. Pembagian dibuat agar campuran skor di tiap bagian mirip satu sama lain.

In [ ]:
kolom = data_bersih.columns.drop("quality")
X, y = data_bersih[kolom], data_bersih["quality"]

model_biasa = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
model_seimbang = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                        class_weight="balanced")

# Data dibagi 5 bagian, tiap bagian bergantian jadi bahan ujian
lima_bagian = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tebakan_biasa = cross_val_predict(model_biasa, X, y, cv=lima_bagian)
tebakan_seimbang = cross_val_predict(model_seimbang, X, y, cv=lima_bagian)
print("Selesai. Setiap wine sudah ditebak oleh kedua model.")

## 7. Evaluasi model

Sekarang kita periksa seberapa bagus tebakannya. Ada dua angka sederhana. Angka pertama adalah persen tebakan yang pas dari seluruh wine. Angka kedua adalah rata-rata kemampuan model menemukan wine dari tiap skor, dengan semua skor dianggap sama penting, termasuk skor yang jarang muncul.

In [ ]:
def ringkas(tebakan):
    return [accuracy_score(y, tebakan) * 100, balanced_accuracy_score(y, tebakan) * 100]

hasil = pd.DataFrame(
    {"Model biasa": ringkas(tebakan_biasa), "Model seimbang": ringkas(tebakan_seimbang)},
    index=["Tebakan yang pas (%)", "Rata-rata kemampuan menemukan tiap skor (%)"]).round(1)
hasil

Model biasa unggul di angka pertama, karena sebagian besar wine memang berskor 5 atau 6 dan itulah yang paling sering ia tebak dengan benar. Model seimbang unggul di angka kedua, sebab ia lebih rajin mencari wine dari skor yang jarang muncul. Grafik berikut memperlihatkan hasilnya skor demi skor.

In [ ]:
skor = sorted(y.unique())
ditemukan = pd.DataFrame({
    "Model biasa": recall_score(y, tebakan_biasa, labels=skor, average=None, zero_division=0) * 100,
    "Model seimbang": recall_score(y, tebakan_seimbang, labels=skor, average=None, zero_division=0) * 100,
}, index=skor)

ax = ditemukan.plot.bar(figsize=(8.5, 4.2), color=[BIRU, ORANYE], width=0.78, rot=0)
for wadah in ax.containers:
    ax.bar_label(wadah, fmt="%.0f", fontsize=9, padding=2)
ax.set(title="Dari 100 wine pada tiap skor, berapa yang berhasil ditemukan?",
       xlabel="Skor kualitas sebenarnya", ylabel="Wine yang ditemukan (dari 100)", ylim=(0, 100))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

for s in (5, 6, 7):
    print(f"Dari 100 wine skor {s}, model biasa menemukan {ditemukan.loc[s, 'Model biasa']:.0f} "
          f"dan model seimbang menemukan {ditemukan.loc[s, 'Model seimbang']:.0f}.")

Tinggi batang menunjukkan berapa dari 100 wine pada skor itu yang berhasil ditemukan. Wine skor 5 ditemukan sama baiknya oleh kedua model. Pada skor 7, model seimbang menemukan jauh lebih banyak wine bagus, tetapi ia lebih sering keliru pada skor 6. Untuk skor 3, 4, dan 8, kedua model masih kesulitan karena contoh pelajarannya terlalu sedikit, yaitu hanya 10, 53, dan 17 wine.

Tabel warna di bawah menjawab pertanyaan lain. Kalau tebakannya salah, salahnya ke mana? Bacalah per baris. Baris adalah skor sebenarnya, kolom adalah skor yang ditebak model, dan angkanya persen dari wine di baris itu. Kotak yang berderet dari kiri atas ke kanan bawah adalah tebakan yang pas.

In [ ]:
tabel = confusion_matrix(y, tebakan_seimbang, labels=skor)
persen = tabel / tabel.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(persen, annot=True, fmt=".0f", cmap="Blues", cbar=False, vmin=0, vmax=100,
            linewidths=2, linecolor="white", xticklabels=skor, yticklabels=skor, ax=ax)
ax.set(title="Salah tebaknya ke mana? (model seimbang, %)",
       xlabel="Skor tebakan model", ylabel="Skor sebenarnya")
plt.tight_layout()
plt.show()

Sebagian besar kesalahan jatuh ke skor tetangga. Wine skor 6, misalnya, lebih sering ditebak 5 atau 7 daripada 3 atau 8. Wine skor 8 yang tadi tampak tidak pernah ditemukan ternyata sebagian besar ditebak skor 7, hanya selisih satu tingkat. Jadi model jarang meleset jauh dan setidaknya menangkap arah kualitas sebuah wine, walau angkanya belum selalu tepat.

Bagian berikutnya menjawab kolom mana yang paling berguna. Caranya, isi satu kolom kita acak-acak sampai tidak punya arti lagi, lalu kita lihat seberapa banyak kemampuan menebak berkurang. Makin besar penurunannya, makin penting kolom itu. Angka mendekati nol berarti kolom itu hampir tidak membantu.

In [ ]:
pengaruh = pd.Series(0.0, index=kolom)
for latih, uji in lima_bagian.split(X, y):
    model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                   class_weight="balanced").fit(X.iloc[latih], y.iloc[latih])
    acak = permutation_importance(model, X.iloc[uji], y.iloc[uji], n_repeats=10,
                                  random_state=42, scoring="balanced_accuracy")
    pengaruh += acak.importances_mean / 5
pengaruh = (pengaruh * 100).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([NAMA[k] for k in pengaruh.index], pengaruh.values, color=BIRU, height=0.65)
for i, nilai in enumerate(pengaruh.values):
    ax.text(max(nilai, 0) + 0.1, i, koma(nilai, 1), va="center", fontsize=10)
ax.set(title="Kolom mana yang paling berguna untuk menebak?",
       xlabel="Penurunan kemampuan menebak jika kolom diacak (%)")
plt.tight_layout()
plt.show()

Kadar alkohol berada jauh di depan, disusul sulfat dan asam mudah menguap. Ketiganya membawa sebagian besar petunjuk. Lima kolom lainnya menyumbang sangat sedikit, dan beberapa hampir nol. Ini cocok dengan grafik di langkah 5, tempat alkohol, sulfat, dan asam mudah menguap memperlihatkan pola yang paling tegas.

### Uji tambahan: wine kembar dan kolom yang dibuang

Dua keputusan di langkah 3 perlu dibuktikan. Sel di bawah menebak skor dengan model biasa pada tiga keadaan data. Pertama, data mentah dengan semua kolom, seperti yang umum dipakai di laporan acuan. Kedua, wine kembar dibuang tetapi semua kolom dipertahankan. Ketiga, wine kembar dan kolom lemah sama-sama dibuang, yaitu data yang kita pakai.

In [ ]:
skenario = {
    "Data mentah, semua kolom": data_mentah,
    "Wine kembar dibuang, semua kolom": data,
    "Wine kembar dan kolom lemah dibuang (dipakai di sini)": data_bersih,
}
for nama, isi in skenario.items():
    tebakan = cross_val_predict(model_biasa, isi.drop(columns="quality"), isi["quality"], cv=lima_bagian)
    print(f"{nama:<55} tebakan pas {accuracy_score(isi['quality'], tebakan):.1%}")

Angka pada data mentah terlihat paling tinggi, padahal sebabnya wine kembar. Model bertemu lagi dengan wine yang sudah ia hafal, jadi ia tampak lebih pintar daripada kenyataannya. Begitu wine kembar dibuang, angkanya turun ke tingkat yang sebenarnya. Baris kedua dan ketiga hampir sama, artinya membuang tiga kolom yang lemah tidak merugikan tebakan.

## 8. Kesimpulan

Data dibersihkan dari wine kembar, yang jumlahnya sekitar 15% dari seluruh baris, dan dari tiga kolom yang hampir tidak berhubungan dengan skor, yaitu sisa gula, belerang bebas, dan pH. Delapan kolom yang tersisa dipakai untuk menebak skor kualitas.

Tiga kolom paling berperan adalah kadar alkohol, sulfat, dan asam mudah menguap. Wine yang dinilai bagus cenderung beralkohol lebih tinggi dan punya lebih banyak sulfat serta asam sitrat. Wine yang dinilai buruk cenderung punya lebih banyak asam cuka dan garam.

Random Forest dapat menebak skor 5 dan 6 dengan cukup baik. Secara keseluruhan sekitar enam dari sepuluh tebakan model biasa tepat. Model seimbang lebih sedikit benar secara keseluruhan, tetapi jauh lebih jeli menemukan wine bagus (skor 7). Pilihlah model seimbang kalau tujuannya mencari wine bagus, dan model biasa kalau yang diinginkan adalah jumlah salah tebak paling sedikit secara keseluruhan.

Skor yang jarang muncul, yaitu 3, 4, dan 8, belum bisa ditebak dengan baik oleh kedua model karena contohnya terlalu sedikit. Angka tebakan pas di notebook ini juga lebih rendah daripada angka sekitar 70% di laporan acuan. Selisihnya muncul karena wine kembar sudah kita buang, dan sebagian angka tinggi di laporan itu berasal dari wine kembar tersebut.

Dua langkah yang layak dicoba berikutnya adalah menambah contoh wine di skor yang langka, dan menyederhanakan skor menjadi tiga tingkat (kurang, sedang, bagus) supaya tiap kelompok punya cukup contoh.